## Pre-Training Strategies: Encoder-Decoder & Decoder-Only Models

This module explores advanced pre-training strategies beyond encoder-only architectures (like BERT), focusing on **Encoder-Decoder** models (BART, T5) and **Decoder-Only** models (the GPT family, LLaMA). Choosing the right architecture depends on the downstream task, memory constraints, and parameter efficiency.

---

### 1. Architectural Setup & Attention Masking

Understanding how attention masks operate is critical to distinguishing these models.

* **Encoder-Only (e.g., BERT):** Bidirectional self-attention. The model sees the entire input sequence simultaneously. Ideal for classification and understanding, but not suited for generation.
* **Decoder-Only (e.g., GPT):** Autoregressive (causal) attention. When processing the $t$-th token, the model can only attend to tokens up to $t-1$. It predicts the next token $X_i$ given $X_1, X_2, \dots, X_{i-1}$. Ideal for generation.
* **Encoder-Decoder (e.g., Vanilla Transformer, BART, T5):**
* **Encoder:** Fully unmasked, bidirectional self-attention over the input sequence.
* **Decoder:** Masked self-attention (autoregressive) + **Cross-attention** where the decoder attends to the encoder's output representations.
* *Trade-off:* Highly capable for sequence-to-sequence tasks (summarization, translation) but requires roughly double the parameters of a decoder-only model for the same stack depth.


* **Prefix Language Model (Prefix-LM):** A hybrid single-stack architecture where the "prefix" (prompt) is processed with bidirectional attention (unmasked), while the subsequent generated tokens are processed autoregressively (masked).

---

### 2. Encoder-Decoder Pre-Training Models

Unlike BERT, which masks tokens and predicts *only* those masked tokens, encoder-decoder models are trained to map corrupted inputs to complete or structured outputs.

#### BART (Bidirectional and AutoRegressive Transformer)

BART combines BERT's bidirectional encoder with GPT's autoregressive decoder.

**Pre-training Objective:** Inject noise into an unlabeled text sequence and train the decoder to reconstruct the original, clean sequence.

* **Input (Encoder):** Corrupted sequence.
* **Output (Decoder):** The *entire* original clean sequence (Contrast this with BERT, which only outputs the masked tokens).

**Corruption Strategies (Noisy Instances):**

1. **Token Masking:** Replace random tokens with `[MASK]`.
2. **Token Deletion:** Remove random tokens entirely. The model must implicitly learn the length and positions of missing tokens.
3. **Permutation:** Shuffle the order of sentences or tokens.
4. **Rotation:** Choose a random token and rotate the document so it starts with that token.
5. **Span Masking (Most Effective):** Mask a contiguous sequence of tokens (e.g., 2 or 3 tokens at once) based on a sampled distribution.

#### T5 (Text-to-Text Transfer Transformer)

Proposed by Google, T5's core philosophy is that **every NLP task can be mapped to a text-to-text format**.

**Task Formulation (The "Prefix" Approach):**
Instead of using task-specific classification heads, T5 relies on a task prefix to guide the model. The output is strictly a text string.

* **Translation:** Input: `Translate English to German: That is good.` $\rightarrow$ Output: `Das ist gut.`
* **Classification (CoLA):** Input: `cola sentence: The course is going well.` $\rightarrow$ Output: `not acceptable`.
* **Regression (STS-B):** Input: `stsb sentence1: The rhino grazed... sentence2: A rhino is grazing...` $\rightarrow$ Output: `3.8`. *(Note: The model is forced to generate the float value as a literal text string).*

**T5 Pre-Training Strategy (Span Corruption):**
T5 uses a span corruption objective, but unlike BART, it does not predict the entire sequence.

* **Sentinel Tokens:** T5 replaces corrupted spans of varying lengths with unique placeholders called Sentinel tokens (e.g., `<X>`, `<Y>`). These are added to the vocabulary.
* **Objective:** The decoder only predicts the masked spans, separated by the sentinel tokens.
* *Example Input:* `Thank you <X> me to your party <Y> week.`
* *Example Output:* `<X> for inviting <Y> last <Z>` (where `<Z>` acts as an end-of-sequence token).

**The C4 Dataset (Colossal Clean Crawled Corpus):**
T5 was trained on a massive web-scraped dataset (Common Crawl). The lecture emphasizes that **cleaning data is as important as volume**. The raw 6 TB corpus was filtered down to 745 GB by removing incomplete sentences, non-English text, HTML/JS tags, and duplicates. Unclean data degraded performance by 2-3%.

---

### 3. Decoder-Only Models

Decoder-only models discard the bidirectional encoder entirely. They rely purely on the autoregressive objective: maximizing the probability of the next token given all previous tokens. This is modeled as a classification problem over a vocabulary of size $V$.

* 117 Million Parameters: GPT-1 (2018)
The first large-scale decoder-only model. Utilized 12 transformer layers, 768-dimensional hidden states, and Byte Pair Encoding (BPE). Trained on a Book Corpus (~7,000 books).


* 1.5 Billion Parameters: GPT-2 (2019)
Introduced the concept of **Zero-Shot Learning**. Vocabulary expanded to 50,000 tokens, context size increased from 512 to 1024, and layer normalization was shifted between sub-blocks. Trained on filtered Common Crawl data (WebText).


* 175 Billion Parameters: GPT-3 (2020)
A massive jump in scale (96 layers, 12,000-dimensional vectors). Proved that massive scaling leads to emergent **In-Context Learning**—the ability to perform tasks just by reading examples in the prompt, eliminating the strict need for task-specific fine-tuning.


* Black Box / Unknown Scale: GPT-4 & Beyond (2023+)
Marked by significant increases in computational cost. Pre-training is now heavily supplemented by **Instruction Fine-Tuning** and **RLHF (Reinforcement Learning from Human Feedback)** to align models with human intent.


**LLaMA (Meta):**
An open-source counterpart to the GPT series. Key architectural differences mentioned include the use of **RoPE (Rotary Position Encoding)** instead of standard absolute positional embeddings, and training on a mix of Common Crawl, GitHub, Wikipedia, and Books data.

---

### 4. NPTEL Practice Questions

**Q1. In the T5 pre-training phase, if the input sentence is "The quick brown fox jumps over the lazy dog", and the words "brown fox" and "lazy" are corrupted, what will the decoder's target output look like?**
a) The quick `[MASK]` jumps over the `[MASK]` dog
b) `<X>` brown fox `<Y>` lazy `<Z>`
c) The quick `<X>` jumps over the `<Y>` dog
d) brown fox lazy

* **Answer:** (b)
* **Explanation:** T5 uses sentinel tokens (`<X>`, `<Y>`) to replace spans in the input. Unlike BART which reconstructs the full sentence, T5's decoder is optimized to generate *only* the corrupted spans preceded by their respective sentinel tokens, ending with a final sentinel token (`<Z>`).

**Q2. Which of the following is a key difference between the pre-training objectives of BERT and BART?**
a) BERT requires labeled data, while BART is self-supervised.
b) BERT predicts only the masked tokens, whereas BART's decoder must predict the entire original sequence.
c) BERT uses autoregressive attention, while BART uses purely bidirectional attention.
d) There is no difference; BART is simply a larger version of BERT.

* **Answer:** (b)
* **Explanation:** BERT's loss is calculated only over the `[MASK]` tokens. BART, being an encoder-decoder, takes the corrupted sequence in the encoder and must autoregressively generate the entire clean sequence in the decoder.

**Q3. Why did the T5 authors utilize "Prefix Language Modeling" as an ablation experiment?**
a) To replace the transformer architecture with an RNN.
b) To test if a single-stack architecture could mimic an encoder-decoder by exposing the prompt bidirectionally and predicting the output autoregressively.
c) To speed up token generation during inference.
d) To strictly separate the C4 dataset into English and non-English tasks.

* **Answer:** (b)
* **Explanation:** Prefix-LM simulates the encoder-decoder attention pattern in a single stack. The prefix (input) uses unmasked (bidirectional) attention, while the target generation uses masked (causal) attention. The authors found it performed on par with a shared-parameter encoder-decoder, though a standard full-parameter encoder-decoder still performed best.

---

### 5. Quick Revision Summary

* **Encoder-Decoder advantage:** Highly effective for seq2seq tasks (translation, summarization) but requires more memory/parameters than decoder-only models.
* **BART:** Corrupts input (mask, delete, permute) $\rightarrow$ predicts the **full clean sequence**.
* **T5 Philosophy:** Every NLP task is a **Text-to-Text** problem. Task formatting relies on instruction prefixes (e.g., `translate English to German:`).
* **T5 Pre-training:** Uses **Sentinel Tokens** to replace spans. Decoder predicts *only* the corrupted spans, improving training speed by reducing generation length.
* **Decoder-Only (GPT/LLaMA):** Relies on predicting the next token $P(X_i \vert{} X_{<i})$.
* **Evolution of GPT:** Moved from fine-tuning (GPT-1) to Zero-Shot (GPT-2) to In-Context Learning (GPT-3) via massive parameter scaling. Modern iterations heavily rely on RLHF and Instruction Tuning.